In [1]:
!git clone https://github.com/Diclo-fenac/option-pricing-using-pinn

Cloning into 'option-pricing-using-pinn'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 140 (delta 74), reused 104 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 164.83 KiB | 856.00 KiB/s, done.
Resolving deltas: 100% (74/74), done.


In [2]:
%cd option-pricing-using-pinn

/home/mium/option-pricing-using-pinn


In [3]:
!pip install -r requirements.txt 

# Model Training

This notebook trains the FNO Option Pricer using the data generated in the previous step.

In [4]:
import os
import sys
import torch

# Ensure the project root is in the path
sys.path.append(os.path.abspath('..'))

from config import Config
from train import train_model

### Weights & Biases Setup (Optional)

Run this cell to set your Weights & Biases API key if you intend to track the training. Remember to also set `config.use_wandb = True` in the configuration cell below.

In [5]:
import wandb

# Uncomment the line below and paste your W&B API key if you want to use it
# wandb.login(key="YOUR_WANDB_API_KEY_HERE")

### Load Configuration

Instantiate the configuration and apply any overrides for training.

In [6]:
config = Config()

# --- Core overrides --- #
config.data_dir = '../data'
config.checkpoint_dir = '../checkpoints'
config.results_dir = '../results'

config.device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.n_epochs = 100
config.batch_size = 16
config.learning_rate = 1e-3

# --- W&B and GCP Settings --- #
config.use_wandb = False                  # Enables W&B logging and auto-naming
config.run_benchmark = True              # Automatically benchmark after training
config.gcp_bucket_name = 'dataset-clut'
config.gcp_service_account_path = '../op-usin-pinn-service.json'
config.gcp_prefix = 'pinns'              # GCS path prefix: gs://bucket/{prefix}/{run_name}/

# --- Checkpointing --- #
config.save_every_n_epochs = 15          # Save epoch checkpoint every N epochs
config.save_latest_every_epoch = 1       # Save checkpoint_latest.pt locally every N epochs
config.resume_from_checkpoint = None     # Set to a path to resume, e.g.:
                                         # config.resume_from_checkpoint = '../checkpoints/fno_model_v1_checkpoint_latest.pt'

# --- GCP Upload Optimization --- #
config.upload_latest_every_n_epochs = 5  # Upload checkpoint_latest to GCP every N epochs
config.upload_best_always = True         # Always upload best.pt to GCP on improvement
config.compress_checkpoints = True       # Compress checkpoints with gzip before upload
config.gcp_storage_class = 'STANDARD'    # 'STANDARD', 'NEARLINE', 'COLDLINE', 'ARCHIVE'
config.gcp_upload_queue_size = 3         # Max pending uploads in queue (backpressure)

# --- WandB Resume --- #
config.wandb_run_id = None               # Set to resume a specific W&B run
config.wandb_resume = 'allow'            # 'allow' | 'must' | None

# Disabling dataloader multiprocessing inside notebooks prevents hanging/crashing
config.num_workers = 0
config.pin_memory = False
config.persistent_workers = False
# -------------------------------- #

In [7]:
import os
from google.cloud import storage

# Create the local data directory if it doesn't exist
os.makedirs(config.data_dir, exist_ok=True)

# Initialize a client using the service account key
client = storage.Client.from_service_account_json(config.gcp_service_account_path)
bucket = client.get_bucket(config.gcp_bucket_name)

# Define the paths for the files in GCS and local filesystem
gcs_train_blob_path = 'data/train.h5'
local_train_file_path = os.path.join(config.data_dir, 'train.h5')

gcs_val_blob_path = 'data/val.h5'
local_val_file_path = os.path.join(config.data_dir, 'val.h5')

# Download train.h5
if not os.path.exists(local_train_file_path):
    print(f"Downloading {gcs_train_blob_path} from GCS to {local_train_file_path}...")
    blob = bucket.blob(gcs_train_blob_path)
    blob.download_to_filename(local_train_file_path)
    print("Download complete for train.h5.")
else:
    print(f"{local_train_file_path} already exists. Skipping download.")

# Download val.h5
if not os.path.exists(local_val_file_path):
    print(f"Downloading {gcs_val_blob_path} from GCS to {local_val_file_path}...")
    blob = bucket.blob(gcs_val_blob_path)
    blob.download_to_filename(local_val_file_path)
    print("Download complete for val.h5.")
else:
    print(f"{local_val_file_path} already exists. Skipping download.")


../data/train.h5 already exists. Skipping download.
../data/val.h5 already exists. Skipping download.


In [8]:
!git pull origin main

From https://github.com/Diclo-fenac/option-pricing-using-pinn
 * branch            main       -> FETCH_HEAD
Already up to date.


In [9]:
import torch
import gc

 # 1. Delete large objects if they exist in the namespace
if 'history' in locals(): del history
if 'trainer' in locals(): del trainer
if 'train_ds' in locals(): del train_ds
if 'val_ds' in locals(): del val_ds

# 2. Force Python garbage collection
gc.collect()

# 3. Empty PyTorch's internal cache
torch.cuda.empty_cache()

 # 4. (Optional) Reset the peak memory stats
torch.cuda.reset_peak_memory_stats()
print(f"GPU Memory Released. Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

RuntimeError: invalid argument to reset_peak_memory_stats

### Resume Training (Optional)

Use these cells to inspect existing checkpoints or resume training from the latest checkpoint.

In [ ]:
# Inspect the latest checkpoint
from train import inspect_checkpoint, load_checkpoint_for_resume
import glob
import os

ckpt_dir = '../checkpoints'
latest_ckpts = glob.glob(os.path.join(ckpt_dir, '*_checkpoint_latest.pt'))
if latest_ckpts:
    path = latest_ckpts[-1]
    info = inspect_checkpoint(path)
    print(f"Found: {path}")
    print(f"Run: {info['run_name']}, Epoch: {info['epoch']}, History: {info['history_length']} epochs")
else:
    print('No checkpoints found.')

In [ ]:
# Resume from latest checkpoint
latest_ckpts = glob.glob(os.path.join(ckpt_dir, '*_checkpoint_latest.pt'))
if latest_ckpts:
    config.resume_from_checkpoint = latest_ckpts[-1]
    print(f"Resuming from: {config.resume_from_checkpoint}")
else:
    print('No checkpoints found. Starting fresh training.')

### Run Training Pipeline

Start the training process. The progress, including RMSE and losses, will be printed below. Checkpoints and loss plots will be saved to the specified directories.

In [10]:
if __name__ == '__main__':
    history = train_model(config)
    print("Training complete!")

Grid: S∈[0.00,600.0] (256), t∈[0,2.0] (64)
PDE interior: 64 × 32 points

Training FNO Option Pricer — Physics-Informed (PDE)
Device: cpu
Parameters: 18,949,697
Train: 8000, Val: 1000
Patience: 25



Epoch   1/100 | Train: 2.112042e+14 | Val RMSE: 3.513329e+01 | Δ RMSE: 3.750950e+00 | Γ RMSE: 2.645906e+01 | LR: 2.80e-04 | 1733s
Uploading ../checkpoints/fno_model_v1_best.pt to gs://dataset-clut/models/fno_model_v1/fno_model_v1_best.pt ...
Upload complete.


KeyboardInterrupt: 